# Week 5 · Thursday Review — Full EDA Pipeline on the Orders Dataset

This notebook demonstrates a complete **Exploratory Data Analysis (EDA) pipeline** using a
synthetic `orders` dataset generated from the exact specification provided for today's
assignment.

The goal of this exercise is not simply to clean a dataset or create a few charts.
Instead, it brings together the skills learned throughout Week 5 into one complete
workflow: understanding the raw data, identifying problems, making justified cleaning
decisions, verifying those changes, visualizing important patterns, and communicating
the final findings clearly.

A key principle throughout this notebook is **diagnose before modifying**. The original
dataset is inspected first so that every data-quality problem can be identified and
documented before any values are changed. Each cleaning decision is then made
individually and supported by a specific reason rather than applying a blanket operation
such as dropping every row containing a missing value.

The notebook is organized into the following eight stages:

1. **Dataset Generation**  
   Generate the `orders` dataset using the exact specification provided in the
   assignment. The dataset intentionally contains several data-quality problems so that
   they can be discovered and handled during the EDA process.

2. **Initial Confirmation**  
   Confirm that the dataset was generated correctly by checking its shape, columns, and
   first few rows. This ensures that the required dataset structure is present before
   beginning the analysis.

3. **Diagnosis**  
   Examine the raw dataset using `.head()`, `.info()`, `.describe()`, `.isna().sum()`,
   `.value_counts()`, and additional targeted checks. The purpose of this stage is to
   identify missing values, inconsistent categories, invalid quantities, unusual prices,
   duplicate records, and any other data-quality issues.

4. **Cleaning**  
   Handle each identified problem using an appropriate and justified method. Missing
   values, inconsistent categories, invalid numerical values, outliers, and duplicates
   are considered separately rather than being fixed with one general operation.

5. **Cleaning Verification**  
   Re-check the cleaned dataset to confirm that the identified problems were actually
   resolved. This includes checking for remaining missing values, duplicates, negative
   quantities, inconsistent categories, and suspicious price values.

6. **Visualization**  
   Use Matplotlib to explore the cleaned data visually. At least three charts are created,
   with each chart selected according to the question being investigated: a distribution,
   a category comparison, and a relationship between numerical variables. All charts are
   created using the `fig, ax = plt.subplots()` approach.

7. **Findings**  
   Convert the analysis and visualizations into clear findings. Each finding is supported
   by a specific number, statistic, or chart rather than being based only on assumptions
   made before analyzing the data.

8. **Technical Summary**  
   Provide a concise explanation of what the dataset contains, the main findings from the
   EDA, the cleaning decisions that were made, and at least one limitation of the
   analysis. The summary is written so that a non-technical reader can understand the
   main results.

Overall, this notebook follows the complete EDA workflow:

**Raw Data → Diagnose → Clean → Verify → Visualize → Find Patterns → Communicate Results**

The objective is to demonstrate not only that the Pandas and Matplotlib commands can be
used correctly, but also that the analysis decisions can be explained, justified, and
reproduced.

## 1. Generate the dataset

Generated exactly from the required spec (seed=42), so the planted problems are the same for everyone.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:

rng = np.random.default_rng(seed=42)
n = 5000

orders = pd.DataFrame({
    "order_id": np.arange(1, n + 1),
    "order_date": pd.date_range("2024-01-01", periods=n, freq="h"),
    "customer_id": rng.integers(1000, 1200, size=n),
    "product_category": rng.choice(
        ["Electronics", "electronics", "Home Goods", "Apparel", "Books"], size=n
    ),
    "quantity": rng.integers(1, 8, size=n),
    "unit_price": rng.normal(45, 20, size=n).round(2),
    "region": rng.choice(["North", "South", "East", "West", None], size=n, p=[0.24, 0.24, 0.24, 0.24, 0.04]),
})

# Introduce the mess, on purpose — do not skip this part
orders.loc[rng.choice(n, 150, replace=False), "customer_id"] = None
orders.loc[rng.choice(n, 30, replace=False), "quantity"] *= -1          # returns, disguised as negative quantity
orders.loc[rng.choice(n, 20, replace=False), "unit_price"] = 4999.99    # data-entry outliers
orders = pd.concat([orders, orders.sample(15, random_state=1)])        # duplicate rows, unannounced


## 2. Initial Confirmation

In [3]:
print(f"Shape: {orders.shape}")
orders.head()

Shape: (5015, 7)


,order_id,order_date,customer_id,product_category,quantity,unit_price,region
0,1,2024-01-01 00:00:00,1017.0,Electronics,3,44.68,West
1,2,2024-01-01 01:00:00,1154.0,Electronics,2,20.69,East
2,3,2024-01-01 02:00:00,1130.0,Apparel,4,41.60,West
3,4,2024-01-01 03:00:00,1087.0,Apparel,5,26.26,South
4,5,2024-01-01 04:00:00,1086.0,Apparel,2,39.45,West


In [4]:
orders.tail()

,order_id,order_date,customer_id,product_category,quantity,unit_price,region
1852,1853,2024-03-18 04:00:00,1090.0,Apparel,4,37.52,South
1185,1186,2024-02-19 09:00:00,1061.0,Books,5,45.70,South
1724,1725,2024-03-12 20:00:00,1106.0,electronics,3,33.71,North
4080,4081,2024-06-19 00:00:00,1180.0,Books,1,46.86,West
3823,3824,2024-06-08 07:00:00,1035.0,Electronics,5,87.65,North


In [5]:
orders.columns

Index(['order_id', 'order_date', 'customer_id', 'product_category', 'quantity',
       'unit_price', 'region'],
      dtype='str')

### Save the raw dataset

Saving the raw, uncleaned `orders` DataFrame to CSV before any diagnosis or cleaning happens — this preserves the original planted problems for reference/reproducibility.

In [6]:
orders.to_csv("orders_raw.csv", index=False)
print(f"Saved {len(orders)} raw rows to orders_raw.csv")

Saved 5015 raw rows to orders_raw.csv
